# INSTALL REQUIRED LIBRARIES

In [22]:
!pip install --upgrade typing_extensions
!pip install --upgrade fastapi
!pip install --upgrade pydantic
!pip install --upgrade gradio


  Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.5.0
    Uninstalling typing_extensions-4.5.0:
      Successfully uninstalled typing_extensions-4.5.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.18.0 requires h5py>=3.11.0, but you have h5py 3.8.0 which is incompatible.
tensorflow-intel 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 1.23.5 which is incompatible.


                                              0.0/103.0 kB ? eta -:--:--
     -------------------------------------- 103.0/103.0 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: fastapi
    Found existing installation: fastapi 0.128.0
    Uninstalling fastapi-0.128.0:
      Successfully uninstalled fastapi-0.128.0
                                              0.0/463.6 kB ? eta -:--:--
     -------                                 92.2/463.6 kB 1.7 MB/s eta 0:00:01
     ---------------------------            337.9/463.6 kB 3.5 MB/s eta 0:00:01
     -------------------------------------- 463.6/463.6 kB 3.6 MB/s eta 0:00:00
                                              0.0/2.0 MB ? eta -:--:--
     --                                       0.1/2.0 MB 7.0 MB/s eta 0:00:01
     --                                       0.1/2.0 MB 7.0 MB/s eta 0:00:01
     --                                       0.1/2.0 MB 944.1 kB/s eta 0:00:02
     ------                                   0.3/2.0 MB 1.6 MB/s

# IMPORT LIBRARIES

In [6]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load dataset
fake = pd.read_csv(r"D:\News_Dataset\Fake.csv")
true = pd.read_csv(r"D:\News_Dataset\True.csv")

fake["label"] = 0
true["label"] = 1

df = pd.concat([fake, true])

# Shuffle dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Combine title + text
df["content"] = df["title"] + " " + df["text"]

X = df["content"]
y = df["label"]

# Train Test Split FIRST (Avoid Data Leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# TF-IDF
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_df=0.7,
    ngram_range=(1,2)
)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

# Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predictions
pred = model.predict(X_test)

# Accuracy
print("Accuracy:", accuracy_score(y_test, pred))

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test, pred))

# Confusion Matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, pred))


# Save model
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))


Accuracy: 0.9839643652561247

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      4710
           1       0.98      0.99      0.98      4270

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980


Confusion Matrix:

[[4619   91]
 [  53 4217]]


# LAUNCN THE LOCAL HOST INTERFACE

In [2]:
import gradio as gr
import pickle

# Load model
model = pickle.load(open("model.pkl", "rb"))
vectorizer = pickle.load(open("vectorizer.pkl", "rb"))

# Prediction function
def predict_news(text):
    if not text.strip():
        return "Please enter some news text."

    vector = vectorizer.transform([text])
    prediction = model.predict(vector)

    if prediction[0] == 1:
        return "✅ This news is Real"
    else:
        return "❌ This news is Fake"


# Interface
interface = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(
        label="Enter News Headline",
        placeholder="Type news headline here..."
    ),
    outputs=gr.Textbox(label="Prediction Result"),
    title="📰 Fake News Detection System",
    description="Enter a news headline below to check whether it is Real or Fake."
)

print("Starting the interface...")
interface.launch(share=False)


Starting the interface...
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
